# Amazon Laptop Scraper

In [12]:
# required lib 

#!pip install beautifulsoup4 requests pandas numpy
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import time
import re

In [ ]:
# Take a number n from user and print sum of first n natural numbers using for loop. Explanation: Initialize sum = 0, loop from 1 to n, add each number to sum.
n = int(input("Enter a number: "))
sum = 0
for i in range(1, n + 1):
    sum += i
print("Sum of first", n, "natural numbers is:", sum)

1
2
4
5
6


In [13]:
# install beautifulsoup4 and requests
#!pip install beautifulsoup4 requests

In [14]:
# take the url of the product page
URL ="https://www.amazon.in/s?k=laptops&crid=36D68A7C7ZL08&sprefix=laptops%2Caps%2C278&ref=nb_sb_noss_1"

In [15]:
# make header to avoid being blocked by amazon

headers = {
    'User-Agent':"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/150.0.0.0 Safari/537.36"
}

In [16]:
# check the status of the request
response = requests.get(URL, headers=headers)

print(response.status_code)

200


In [17]:
# check the content 
print(response.content)

b'<!DOCTYPE html><html><head> <meta charset="utf-8"> <meta name="viewport" content="width=device-width, initial-scale=1, shrink-to-fit=no"> <meta http-equiv="refresh" content="5; URL=\'/s?k=laptops&amp;crid=36D68A7C7ZL08&amp;sprefix=laptops%2Caps%2C278&amp;ref=nb_sb_noss_1&bm-verify=AAQAAAAN_____8jNNFnVM7hYIj_aDYJhazpudZ2deEN1D7judovKzydU253jeNvSCkNEtk98QOWSPuhZnCUDMrweCG_jXMkqWA5M89byBljFtjxhelp2Y0EQ0JdMSznM8E8sYTGpO2Bp-LjyxsQKaFEQSPGwzwkRqv_WTy7a9no0N3elQnm_xRqeQm9m2837iQf_jBhgubbZ9E7mCnOhJMlpc-FaT5HyhRxQ-pWRh0qc_alYkLYNvxf_6fMeyDXYkEGxNrhLOc1R7Fkf7trFyFzBIhQB0muXlbom6je1pU_VxZJiV4KrPtVUgRxr_7dtrKrGYyKrg88TRK6pOpyMsQDZGDIyZ_5YnFhdo0PnAmc-gyfRNts\'" /><title>&nbsp;</title><script> var i = 1784040825; var j = i + Number("3128" + "7581"); </script> </head> <noscript> <iframe style="border: none; height: 100%; width: 100%;" src=""></iframe></noscript><body> <iframe style="border: none; width: 100vw; height: 100vh;" src="https://m.media-amazon.com/images/S/sash/6Uh4bsAwUkB3vJb.gif"> </ifr

In [18]:
# convert the content to beautifulsoup object
soup = BeautifulSoup(response.content, 'html.parser')

# print the soup content
print(soup.prettify())

<!DOCTYPE html>
<html>
 <head>
  <meta charset="utf-8"/>
  <meta content="width=device-width, initial-scale=1, shrink-to-fit=no" name="viewport"/>
  <meta content="5; URL='/s?k=laptops&amp;crid=36D68A7C7ZL08&amp;sprefix=laptops%2Caps%2C278&amp;ref=nb_sb_noss_1&amp;bm-verify=AAQAAAAN_____8jNNFnVM7hYIj_aDYJhazpudZ2deEN1D7judovKzydU253jeNvSCkNEtk98QOWSPuhZnCUDMrweCG_jXMkqWA5M89byBljFtjxhelp2Y0EQ0JdMSznM8E8sYTGpO2Bp-LjyxsQKaFEQSPGwzwkRqv_WTy7a9no0N3elQnm_xRqeQm9m2837iQf_jBhgubbZ9E7mCnOhJMlpc-FaT5HyhRxQ-pWRh0qc_alYkLYNvxf_6fMeyDXYkEGxNrhLOc1R7Fkf7trFyFzBIhQB0muXlbom6je1pU_VxZJiV4KrPtVUgRxr_7dtrKrGYyKrg88TRK6pOpyMsQDZGDIyZ_5YnFhdo0PnAmc-gyfRNts'" http-equiv="refresh"/>
  <title>
  </title>
  <script>
   var i = 1784040825; var j = i + Number("3128" + "7581");
  </script>
 </head>
 <noscript>
  <iframe src="" style="border: none; height: 100%; width: 100%;">
  </iframe>
 </noscript>
 <body>
  <iframe src="https://m.media-amazon.com/images/S/sash/6Uh4bsAwUkB3vJb.gif" style="border: none; width

In [19]:
# make the empty list to store the data
data = []
print("Data list created")

Data list created


In [20]:
data = []
seen_titles = set()

for page in range(1, 2):

    # make the parameter for the request
    params = {
        'k': 'laptops',
        'page': page
    }

    # get the response from the request
    response = requests.get(URL, headers=headers, params=params, timeout=15)
    response.raise_for_status()

    # convert the content to beautifulsoup object
    soup = BeautifulSoup(response.content, 'html.parser')

    # find all Amazon product result cards
    product_cards = soup.select('div[data-component-type="s-search-result"]')

    # make the for loop to get the data from the product cards
    for product in product_cards:

        # step 1: extract the title
        title_tag = product.select_one('h2 span')
        if not title_tag:
            continue

        title = title_tag.get_text(" ", strip=True)
        title = ' '.join(title.split())

        # Skip irrelevant text and page headings
        lower_title = title.lower()
        if not title or len(title) < 10:
            continue
        if 'results for' in lower_title:
            continue
        if any(skip in lower_title for skip in [
            'trending now', 'best sellers', 'new releases', 'shop by', 'related searches'
        ]):
            continue
        if title in seen_titles:
            continue

        seen_titles.add(title)
    

        # Step 2: extract the price
        price_tag = product.select_one('span.a-price-whole')
        if price_tag:
            price = price_tag.get_text(" ", strip=True)
        else:
            price = None

        # Step 3 : Extract the brand
        match = re.search(r"^([A-Za-z]+)", title)
        brand = match.group(1) if match else None

        # Step 4: Extract the Ram 
        match = re.search(r"(\d+GB|RAM\s*(\d+GB)?|DDR\d?\s*(\d+GB)?|LPDDR\d?\s*(\d+GB)?)", title, re.IGNORECASE)
        ram = match.group(1) if match else None
        
        # Step 5: Extract the storage
        match = re.search(r"(\d+)\s*(GB|TB)\s*(?:SSD|Storage|HDD)", title, re.IGNORECASE)
        ssd_storage = f"{match.group(1)}{match.group(2)}" if match else "N/A"

        #Step 6 : Extract the Color from the title using regex
        match = re.search(r"\b(Black|White|Silver|Gray|Grey|Red|Blue|Green|Yellow|Pink|Purple|Gold|Bronze|Rose Gold|Indigo|Glacier)\b", title, re.IGNORECASE)
        color = match.group(1) if match else "N/A"
        
        # Step 7: Extract the processor (Intel, AMD, Apple M/A chip, Snapdragon, MediaTek, etc)
        processor = "N/A"
        # Try to match Apple M series (M1, M2, M3, M4, M5, etc.)
        match = re.search(r"Apple\s+M(\d+)", title, re.IGNORECASE)
        if match:
            processor = f"Apple M{match.group(1)}"
        else:
            # Try to match Apple A series (A18, A17, A16, etc.)
            match = re.search(r"Apple\s+A(\d+)", title, re.IGNORECASE)
            if match:
                processor = f"Apple A{match.group(1)}"
            else:
                # Try other processor keywords
                processor_keywords = ["Intel", "AMD", "Snapdragon", "MediaTek", "Celeron"]
                for keyword in processor_keywords:
                    if re.search(rf"\b{keyword}\b", title, re.IGNORECASE):
                        processor = keyword
                        break

        # append the data to the list
        data.append({
            'title': title,
            'price': price,
            'brand': brand,
            'ram': ram,
            'storage': ssd_storage,
            'color': color,
            'processor': processor
        })


In [21]:
# scrap data structure formate 

for product in data:
    print(f"Title: {product['title']}")
    print(f"Price: {product['price']}")
    print(f"Brand: {product['brand']}")
    print(f"Ram: {product['ram']}")
    print(f"Storage: {product['storage']}")
    print(f"Color: {product['color']}")
    print(f"Processor: {product['processor']}")
    print("-" * 40)

Title: ASUS Vivobook 16, Smartchoice, Intel Core Ultra 5 Series 2, 16GB RAM, 512GB SSD, FHD+ 16", Windows 11, Office Home 2024, Silver, 1.88kg, X1607CA-MB142WS, Intel iGPU, M365 Basic (1Year), AI Laptop
Price: 73,990
Brand: ASUS
Ram: 16GB
Storage: 512GB
Color: Silver
Processor: Intel
----------------------------------------
Title: ASUS Vivobook 16 (2026),Smartchoice,Intel Core Ultra 5 325 (Series 3), Intel iGPU,16GB RAM,512GB SSD,FHD+, 16"(40 cm),Win11,M365 Basic(1Y),Office 24,Quiet Blue,1.88 kg,X1607AA-MB037WS, Copilot+ AI PC
Price: 94,990
Brand: ASUS
Ram: 16GB
Storage: 512GB
Color: Blue
Processor: Intel
----------------------------------------
Title: ASUS Vivobook 15, Smartchoice,Intel Core i3 13th Gen 1315U, 12GB RAM, 512GB SSD, FHD 15.6", Windows 11, Office 2024, Cool Silver, 1.7Kg, X1504VA-BQ331WS, Intel UHD iGPU, Thin & Light, 42Whrs Laptop
Price: 48,990
Brand: ASUS
Ram: 12GB
Storage: 512GB
Color: Silver
Processor: Intel
----------------------------------------
Title: HP 15, AMD 

In [11]:
# length of the data list
print(f"Number of products scraped: {len(data)}")

Number of products scraped: 0
